# 🚗 02. YOLO11 Detection + ByteTrack + TrajectoryManager

**Tuần 2 (08/09 – 14/09)** — Kaggle Notebook

**Mục tiêu:** Detection + Tracking chạy được trên video mẫu, output ra trajectory cơ bản.

## Pipeline tuần này
```
Video đầu vào
    ↓
[YOLO11s pretrained COCO] — detect: car, motorcycle, bus, truck, person, bicycle
    ↓
[ByteTrack] — gắn track ID nhất quán theo thời gian
    ↓
[TrajectoryManager] — lưu quỹ đạo, tính velocity / acceleration
    ↓
trajectories.csv + trajectories.json  ← input cho Tuần 3
```

---
**GPU quota tuần này:** ~5h Kaggle (detection/tracking không nặng)

| Cell | Nội dung | Cần GPU? |
|------|----------|----------|
| 1–2  | Setup môi trường | Không |
| 3–5  | Detection YOLO11s | ✅ Có |
| 6–8  | Tracking ByteTrack | ✅ Có |
| 9–11 | TrajectoryManager | Không |
| 12   | Visualize trajectory | Không |
| 13–14| Export + Upload Drive | Không |

## ⚙️ Cell 1 — Setup môi trường

In [ ]:
import os, sys, json, time, random
from collections import defaultdict
import numpy as np

# ── Phát hiện môi trường ──────────────────────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB  = 'google.colab' in sys.modules
print(f'Environment: {"Kaggle" if IS_KAGGLE else "Colab" if IS_COLAB else "Local"}')

# ── Thư mục làm việc ─────────────────────────────────────────────────────
if IS_KAGGLE:
    WORK_DIR   = '/kaggle/working/week2_output'
    # Dataset input (nếu đã upload video lên Kaggle Dataset)
    INPUT_DIR  = '/kaggle/input'
elif IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive/accident_detection'
    WORK_DIR   = f'{DRIVE_BASE}/results/week2_output'
    INPUT_DIR  = f'{DRIVE_BASE}/datasets'
else:
    WORK_DIR  = 'results/week2_output'
    INPUT_DIR = 'data'

os.makedirs(WORK_DIR, exist_ok=True)
print(f'✅ Work dir: {WORK_DIR}')

## ⚙️ Cell 2 — Cài thư viện + kiểm tra GPU

In [ ]:
# Cài ultralytics (YOLO11 + ByteTrack tích hợp sẵn)
!pip install -q ultralytics

import torch
from ultralytics import YOLO
import ultralytics

print(f'Ultralytics : {ultralytics.__version__}')
print(f'PyTorch     : {torch.__version__}')
print(f'CUDA avail  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU         : {gpu.name}')
    print(f'VRAM        : {gpu.total_memory / 1e9:.1f} GB')
else:
    print('⚠️  Không tìm thấy GPU — hãy bật Accelerator trong Settings!')

## 🎯 Cell 3 — Cấu hình đường dẫn video

> **Cách lấy video mẫu:**
> - Trên Kaggle: upload video lên dataset riêng → mount vào `/kaggle/input/`
> - Trên Colab: đặt video vào `MyDrive/accident_detection/results/sample_videos/`
> - Hoặc tải nhanh clip mẫu từ CADP/CCD dataset đã có

In [ ]:
# ── ĐỔI ĐƯỜNG DẪN VIDEO CỦA BẠN VÀO ĐÂY ────────────────────────────────
if IS_KAGGLE:
    # Kaggle: thay 'your-dataset' bằng tên Kaggle Dataset bạn đã upload
    VIDEO_PATH = '/kaggle/input/your-dataset/sample_traffic.mp4'
elif IS_COLAB:
    VIDEO_PATH = f'{DRIVE_BASE}/results/sample_videos/sample_traffic.mp4'
else:
    VIDEO_PATH = 'demo/sample_videos/sample_traffic.mp4'
# ─────────────────────────────────────────────────────────────────────────

if not os.path.exists(VIDEO_PATH):
    print(f'⚠️  Video không tồn tại: {VIDEO_PATH}')
    print('   → Hãy upload video và cập nhật VIDEO_PATH bên trên.')
else:
    import cv2
    cap = cv2.VideoCapture(VIDEO_PATH)
    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    print(f'✅ Video: {os.path.basename(VIDEO_PATH)}')
    print(f'   {width}×{height}  {fps:.1f}fps  {frames} frames  ({frames/fps:.1f}s)')

## 🔍 Cell 4 — Load YOLO11s pretrained (COCO)

**Tại sao YOLO11s?**
| Model | Params | mAP COCO | Speed | Dùng khi |
|-------|--------|----------|-------|----------|
| yolo11n | 2.6M | 39.5 | Rất nhanh | Edge, demo nhanh |
| **yolo11s** | **9.4M** | **47.0** | **Nhanh** | **✅ Tuần 2-4** |
| yolo11m | 20.1M | 51.5 | TB | Cần accuracy cao |

→ yolo11s: cân bằng tốt nhất, đủ tốt cho prototype, inference nhanh trên T4.

In [ ]:
# Tải yolo11s.pt (tự động download từ Ultralytics nếu chưa có)
model = YOLO('yolo11s.pt')

# Class IDs cần giữ (COCO pretrained)
VEHICLE_CLASSES = [0, 1, 2, 3, 5, 7]  # person, bicycle, car, motorcycle, bus, truck
CLASS_NAMES = {
    0: 'person', 1: 'bicycle', 2: 'car',
    3: 'motorcycle', 5: 'bus', 7: 'truck'
}

print('✅ YOLO11s loaded!')
print(f'   Classes kept: {CLASS_NAMES}')

## 🔍 Cell 5 — Test detection trên vài frame mẫu

In [ ]:
import cv2
import matplotlib.pyplot as plt
from collections import Counter

# Chạy detection trên tối đa 5 giây đầu video để test nhanh
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
sample_frames = int(fps * 5)  # 5 giây đầu

print(f'🔍 Testing detection on first {sample_frames} frames...')
all_class_counts = Counter()
preview_frames = []

for f_idx in range(sample_frames):
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(
        source=frame,
        conf=0.4,
        classes=VEHICLE_CLASSES,
        verbose=False
    )

    r = results[0]
    if r.boxes is not None:
        for cls in r.boxes.cls.cpu().numpy().astype(int):
            all_class_counts[CLASS_NAMES.get(cls, str(cls))] += 1

    # Lưu 3 frame để preview
    if f_idx in [0, sample_frames // 2, sample_frames - 1]:
        annotated = r.plot()
        preview_frames.append(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))

cap.release()

# Hiển thị kết quả
print(f'\n📊 Phân bố class trong {sample_frames} frames đầu:')
for cls, cnt in all_class_counts.most_common():
    print(f'   {cls:12s}: {cnt:5d} detections')

# Plot preview
if preview_frames:
    fig, axes = plt.subplots(1, len(preview_frames), figsize=(18, 5))
    if len(preview_frames) == 1:
        axes = [axes]
    for ax, img in zip(axes, preview_frames):
        ax.imshow(img)
        ax.axis('off')
    plt.suptitle('YOLO11s Detection Preview (frame 0, middle, end)', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{WORK_DIR}/detection_preview.png', dpi=100, bbox_inches='tight')
    plt.show()
    print(f'\n✅ Preview saved: {WORK_DIR}/detection_preview.png')

## 📡 Cell 6 — Cấu hình ByteTrack

**Tham số tùy chỉnh cho giao thông Việt Nam:**

| Tham số | Mặc định | Tùy chỉnh | Lý do |
|---------|----------|-----------|-------|
| `track_high_thresh` | 0.5 | **0.4** | Xe máy nhỏ thường có conf thấp |
| `new_track_thresh` | 0.6 | **0.5** | Khởi tạo track sớm hơn |
| `track_buffer` | 30 | **60** | Giữ track lâu hơn khi bị che khuất |
| `track_low_thresh` | 0.1 | 0.1 | Giữ mặc định |
| `match_thresh` | 0.8 | 0.8 | Giữ mặc định |

In [ ]:
import yaml

# Tạo file ByteTrack config tùy chỉnh
bytetrack_cfg = {
    'tracker_type'     : 'bytetrack',
    'track_high_thresh': 0.4,   # hạ từ 0.5 cho xe máy nhỏ
    'track_low_thresh' : 0.1,
    'new_track_thresh' : 0.5,   # hạ từ 0.6
    'track_buffer'     : 60,    # giữ track 60 frame khi mất (~2s ở 30fps)
    'match_thresh'     : 0.8,
    'fuse_score'       : True,
}

tracker_yaml = f'{WORK_DIR}/bytetrack_custom.yaml'
with open(tracker_yaml, 'w') as f:
    yaml.dump(bytetrack_cfg, f, default_flow_style=False)

print(f'✅ ByteTrack config saved: {tracker_yaml}')
print(yaml.dump(bytetrack_cfg, default_flow_style=False))

## 📡 Cell 7 — Chạy YOLO11 + ByteTrack tracking

> ⏱️ **Thời gian ước tính:** ~1 phút cho video 30s trên T4 GPU.

In [ ]:
import time

# Thu thập raw tracking data (frame_id → list of tracks)
raw_tracking = []   # list of {frame_id, tracks: [{track_id, bbox, class_id, confidence}]}

t0 = time.time()
print('▶️  Running YOLO11s + ByteTrack...')

# stream=True: xử lý từng frame, tiết kiệm RAM (quan trọng với video dài)
results_gen = model.track(
    source=VIDEO_PATH,
    persist=True,           # giữ track ID liên tục giữa các frame
    tracker=tracker_yaml,   # dùng config tùy chỉnh
    conf=0.4,
    iou=0.5,
    classes=VEHICLE_CLASSES,
    imgsz=640,
    stream=True,            # ← quan trọng: không load toàn bộ vào RAM
    verbose=False,
)

for frame_idx, result in enumerate(results_gen):
    frame_data = {'frame_id': frame_idx, 'tracks': []}

    if result.boxes is not None and result.boxes.id is not None:
        tids   = result.boxes.id.cpu().numpy().astype(int)
        bboxes = result.boxes.xyxy.cpu().numpy()
        clss   = result.boxes.cls.cpu().numpy().astype(int)
        cfs    = result.boxes.conf.cpu().numpy()

        for tid, bbox, cls, cf in zip(tids, bboxes, clss, cfs):
            frame_data['tracks'].append({
                'track_id'  : int(tid),
                'bbox'      : bbox.tolist(),
                'class_id'  : int(cls),
                'confidence': round(float(cf), 4),
            })

    raw_tracking.append(frame_data)

    if frame_idx % 150 == 0 and frame_idx > 0:
        elapsed = time.time() - t0
        active  = len(frame_data['tracks'])
        print(f'  Frame {frame_idx:5d}  active_tracks={active:3d}  elapsed={elapsed:.1f}s')

elapsed = time.time() - t0
total_frames = len(raw_tracking)
print(f'\n✅ Tracking done!')
print(f'   Frames processed: {total_frames}')
print(f'   Total time      : {elapsed:.1f}s  ({total_frames/elapsed:.1f} FPS effective)')
print(f'   Unique track IDs: {len({t["track_id"] for f in raw_tracking for t in f["tracks"]})}')

## 📡 Cell 8 — Lưu raw tracking data

In [ ]:
import json

raw_json_path = f'{WORK_DIR}/tracking_raw.json'
with open(raw_json_path, 'w') as f:
    json.dump(raw_tracking, f)

size_kb = os.path.getsize(raw_json_path) / 1024
print(f'✅ Raw tracking saved: {raw_json_path}  ({size_kb:.0f} KB)')

# Preview frame 0
print(f'\n📋 Sample (frame 0):')
print(json.dumps(raw_tracking[0], indent=2))

## 🗃️ Cell 9 — TrajectoryManager: build từ raw tracking

In [ ]:
# Import TrajectoryManager từ src
# Trên Kaggle: cần clone repo trước hoặc copy code vào đây
# ── Option A: import từ repo ─────────────────────────────────────────────
# !git clone https://github.com/YOUR_REPO/accident-detection.git repo
# sys.path.insert(0, 'repo')
# from src.tracking.trajectory_manager import TrajectoryManager

# ── Option B: inline class (paste code từ src/tracking/trajectory_manager.py)
# Xem file: https://github.com/YOUR_REPO/accident-detection/blob/main/src/tracking/trajectory_manager.py

# ── Dùng tạm class inline để notebook tự chứa ───────────────────────────
from collections import deque
from typing import Dict, List, Tuple, Optional, Any
import numpy as np, json, csv, os

COCO_VEHICLE_CLASSES = {
    0:'person', 1:'bicycle', 2:'car', 3:'motorcycle', 5:'bus', 7:'truck'
}

# [Class TrajectoryManager đầy đủ — xem src/tracking/trajectory_manager.py]
# Để notebook ngắn gọn, ta import trực tiếp:
try:
    sys.path.insert(0, '/kaggle/working/repo' if IS_KAGGLE else '.')
    from src.tracking.trajectory_manager import TrajectoryManager
    print('✅ TrajectoryManager imported from src/')
except ImportError:
    print('⚠️  Không import được src/ — hãy clone repo hoặc paste class inline.')
    print('   Xem: src/tracking/trajectory_manager.py trong repo.')

## 🗃️ Cell 10 — Populate TrajectoryManager

In [ ]:
manager = TrajectoryManager(max_history=90)

for frame_data in raw_tracking:
    fid    = frame_data['frame_id']
    tracks = frame_data['tracks']

    if tracks:
        tids   = np.array([t['track_id']   for t in tracks], dtype=np.int32)
        bboxes = np.array([t['bbox']        for t in tracks], dtype=np.float32)
        clss   = np.array([t['class_id']    for t in tracks], dtype=np.int32)
        cfs    = np.array([t['confidence']  for t in tracks], dtype=np.float32)
        manager.update(fid, tids, bboxes, clss, cfs)

    # Cleanup định kỳ
    if fid > 0 and fid % 300 == 0:
        manager.cleanup_old_tracks(fid, max_age=90)

# Thống kê
summary = manager.get_summary()
print('📊 TrajectoryManager Summary:')
for k, v in summary.items():
    print(f'   {k:25s}: {v}')

## 🗃️ Cell 11 — Phân tích chuyển động (velocity, acceleration)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

active_tracks = manager.get_active_tracks(min_length=20)
print(f'✅ Active tracks (≥20 frames): {len(active_tracks)}')

rows = []
for tid in active_tracks:
    speed  = manager.get_speed(tid)
    accel  = manager.get_acceleration(tid)
    dtheta = manager.get_direction_changes(tid)
    last   = manager.get_last_state(tid)

    if speed is not None:
        rows.append({
            'track_id'         : tid,
            'class'            : COCO_VEHICLE_CLASSES.get(last['class_id'], '?'),
            'length_frames'    : len(manager.tracks[tid]),
            'avg_speed_px_f'   : round(np.mean(speed), 2),
            'max_speed_px_f'   : round(np.max(speed), 2),
            'max_decel_px_f2'  : round(float(np.min(np.diff(speed))), 2) if len(speed) > 1 else 0,
            'max_dir_change_deg': round(float(np.max(np.abs(dtheta))), 1) if dtheta is not None and len(dtheta) else 0,
        })

df = pd.DataFrame(rows).sort_values('avg_speed_px_f', ascending=False)
print('\n📈 Track Analysis (top 15):')
print(df.head(15).to_string(index=False))

# Plot speed profiles
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Speed histogram
axes[0].hist(df['avg_speed_px_f'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Avg Speed (pixel/frame)')
axes[0].set_ylabel('Number of Tracks')
axes[0].set_title('Speed Distribution of All Tracks')

# Theo dõi tốc độ 1 track cụ thể
if active_tracks:
    sample_tid = active_tracks[0]
    speed_arr  = manager.get_speed(sample_tid)
    if speed_arr is not None:
        axes[1].plot(speed_arr, color='tomato')
        axes[1].set_xlabel('Frame (relative)')
        axes[1].set_ylabel('Speed (px/frame)')
        axes[1].set_title(f'Speed Profile — Track #{sample_tid}')
        axes[1].axhline(np.mean(speed_arr), color='gray', ls='--', label='mean')
        axes[1].legend()

plt.tight_layout()
plt.savefig(f'{WORK_DIR}/speed_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'\n✅ Speed analysis saved: {WORK_DIR}/speed_analysis.png')

## 🎬 Cell 12 — Visualize trajectory trên video

> ⏱️ **Thời gian ước tính:** ~30–60 giây cho video 30s (CPU, không cần GPU).

In [ ]:
import cv2, random

TAIL_FRAMES = 30   # số frame lịch sử vẽ đuôi trajectory

def get_color(track_id):
    rng = random.Random(track_id * 2654435761)
    return (rng.randint(80,255), rng.randint(80,255), rng.randint(80,255))  # BGR

# Tổ chức lại history theo frame_id để lookup nhanh
frame_lookup = defaultdict(list)
for frame_data in raw_tracking:
    for t in frame_data['tracks']:
        frame_lookup[frame_data['frame_id']].append(t)

# Build center history cho mỗi track
center_history = {}   # tid → [(frame_id, (cx, cy))]
for tid, history in manager.tracks.items():
    center_history[tid] = [
        (s['frame_id'], (s['center'][0], s['center'][1]))
        for s in history
    ]

# Video writer
cap = cv2.VideoCapture(VIDEO_PATH)
fps_vid = cap.get(cv2.CAP_PROP_FPS) or 30
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

out_path = f'{WORK_DIR}/trajectory_visualization.mp4'
writer   = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps_vid, (W, H))

print(f'🎬 Rendering trajectory video...')
t0 = time.time()

for frame_idx in range(total):
    ret, frame = cap.read()
    if not ret:
        break

    # 1. Vẽ đuôi trajectory (mờ dần về quá khứ)
    for tid, hist in center_history.items():
        tail = [(f,c) for f,c in hist if frame_idx - TAIL_FRAMES <= f <= frame_idx]
        color = get_color(tid)
        for i in range(1, len(tail)):
            alpha = i / len(tail)
            c = tuple(int(v * alpha) for v in color)
            p1 = (int(tail[i-1][1][0]), int(tail[i-1][1][1]))
            p2 = (int(tail[i][1][0]),   int(tail[i][1][1]))
            cv2.line(frame, p1, p2, c, 2, cv2.LINE_AA)
        if tail:
            cur = (int(tail[-1][1][0]), int(tail[-1][1][1]))
            cv2.circle(frame, cur, 4, color, -1)

    # 2. Vẽ bbox + label cho frame hiện tại
    for t in frame_lookup.get(frame_idx, []):
        tid, bbox = t['track_id'], t['bbox']
        color = get_color(tid)
        x1,y1,x2,y2 = map(int, bbox)
        cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
        cls_name = CLASS_NAMES.get(t['class_id'], str(t['class_id']))
        label = f"{cls_name} #{tid}"
        cv2.putText(frame, label, (x1, max(y1-6, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)

    # 3. Overlay info
    n_active = len(frame_lookup.get(frame_idx, []))
    cv2.putText(frame, f'Frame: {frame_idx}  Tracks: {n_active}',
                (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

    writer.write(frame)

cap.release()
writer.release()
print(f'✅ Video saved: {out_path}  ({time.time()-t0:.1f}s)')

## 💾 Cell 13 — Export trajectories.csv + trajectories.json

In [ ]:
# Export CSV
csv_path = f'{WORK_DIR}/trajectories.csv'
manager.export_csv(csv_path)

# Export JSON
json_path = f'{WORK_DIR}/trajectories.json'
manager.export_json(json_path)

# Đọc lại CSV preview
import pandas as pd
df_full = pd.read_csv(csv_path)
print(f'\n📋 trajectories.csv: {len(df_full)} rows × {len(df_full.columns)} cols')
print(df_full.head(8).to_string(index=False))

print(f'\n📁 Files ready for Week 3:')
for path in [csv_path, json_path, raw_json_path]:
    size = os.path.getsize(path) / 1024
    print(f'   {os.path.basename(path):35s}  {size:.0f} KB')

## ☁️ Cell 14 — Upload kết quả về Google Drive

> Thực hiện sau khi đã commit notebook (Save & Run All) để lấy output.

In [ ]:
# ── CÁCH 1: Dùng pydrive2 (Colab có sẵn auth) ─────────────────────────
if IS_COLAB:
    print('Colab: file đã lưu trực tiếp vào Drive. Không cần upload thêm.')
    print(f'  ✅ {WORK_DIR}')

# ── CÁCH 2: Kaggle — tạo Output Dataset ────────────────────────────────
elif IS_KAGGLE:
    print('Kaggle: Để lưu kết quả về Drive, dùng một trong 2 cách:')
    print()
    print('Cách A (khuyến nghị) — Kaggle Output Dataset:')
    print('  1. Click "Save Version" → "Save & Run All (Commit)"')
    print('  2. Sau khi xong, vào Output tab → "+ New Dataset"')
    print('  3. Dataset sẽ được mount lại ở notebook khác: /kaggle/input/<tên>/')
    print()
    print('Cách B — Upload thủ công qua rclone:')
    print('  !curl https://rclone.org/install.sh | sudo bash -q')
    print('  !rclone copy /kaggle/working/week2_output/ gdrive:accident_detection/results/week2_output/ --progress')
    print()
    print(f'📁 Files tại: {WORK_DIR}')
    for fn in os.listdir(WORK_DIR):
        fpath = os.path.join(WORK_DIR, fn)
        size  = os.path.getsize(fpath) / 1024
        print(f'   {fn:40s}  {size:.0f} KB')

---

## ✅ Checklist Deliverables Tuần 2

| # | Item | File | Status |
|---|------|------|--------|
| 1 | YOLO11s detection chạy được | `detection_preview.png` | ☐ |
| 2 | ByteTrack tracking hoạt động | `tracking_raw.json` | ☐ |
| 3 | `TrajectoryManager` đầy đủ | `src/tracking/trajectory_manager.py` | ☐ |
| 4 | Phân tích speed/accel | `speed_analysis.png` | ☐ |
| 5 | Video demo trajectory | `trajectory_visualization.mp4` | ☐ |
| 6 | Export CSV + JSON | `trajectories.csv`, `trajectories.json` | ☐ |
| 7 | Sync lên Google Drive / Kaggle Dataset | — | ☐ |

---

**Tuần tiếp theo:** Tuần 3 — Rule-Based Baseline sẽ đọc `trajectories.json`
và phát hiện tai nạn bằng các luật: giảm tốc đột ngột, thay đổi hướng, IoU chồng lấn.